# Optimization Techniques Lab Examination
**Name:** Dhruv Lad  
**Student ID:** 241070037  
**Subject:** Optimization Techniques (OT)  
**Question:** Q4 – ML + Quantum + Parallel Computing

## Setup
Installs only the packages that are missing (`dimod`, `dwave-neal`). NumPy, pandas and scikit-learn are preinstalled on Colab.

In [1]:
import importlib.util, subprocess, sys

# Install only if missing (simulated annealing runs on CPU, no quantum hardware)
if importlib.util.find_spec("dimod") is None or importlib.util.find_spec("neal") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dimod", "dwave-neal"])

import time, itertools, os, importlib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import dimod, neal

SEED = 37
N = 10          # number of nodes
HALF = 5        # nodes per island
print("Setup complete.")

Setup complete.


# Part A: Machine Learning (1 Mark)
Synthetic data: **line age, load factor, weather exposure, line length**. The failure label is drawn from a Bernoulli distribution whose probability is a logistic function of the (standardised) features, so the labels are random but reproducible and depend meaningfully on the features.

**Edge-weight formula:** for a link between nodes *i* and *j*

$$w_{ij} = 1 + 10\cdot\frac{p_i + p_j}{2}$$

where $p_i$ is the predicted failure probability of node *i*. Links joining failure-prone nodes are more critical/costly to cut, and the constant 1 keeps every weight strictly positive.

In [2]:
rng = np.random.default_rng(SEED)
n_samples = 600

# --- synthetic features ---
line_age     = rng.uniform(1, 50, n_samples)      # years
load_factor  = rng.uniform(0.3, 1.0, n_samples)   # fraction of capacity
weather_exp  = rng.uniform(0.0, 1.0, n_samples)   # exposure index
line_length  = rng.uniform(5, 100, n_samples)     # km
X = np.column_stack([line_age, load_factor, weather_exp, line_length])
feature_names = ["line_age", "load_factor", "weather_exposure", "line_length"]

# --- probabilistic labels: logistic function of standardised features ---
mu, sd = X.mean(axis=0), X.std(axis=0)
Z = (X - mu) / sd
logit = -0.5 + 1.0*Z[:, 0] + 0.9*Z[:, 1] + 0.8*Z[:, 2] + 0.5*Z[:, 3]
p_true = 1.0 / (1.0 + np.exp(-logit))
y = (rng.random(n_samples) < p_true).astype(int)

# --- 70/30 split and model ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
clf = GradientBoostingClassifier(random_state=SEED)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print(f"Samples: {n_samples}, failure rate: {y.mean():.2f}, train/test: {len(y_tr)}/{len(y_te)}")
print(f"Accuracy : {accuracy_score(y_te, y_pred):.3f}")
print(f"Precision: {precision_score(y_te, y_pred):.3f}")
print(f"Recall   : {recall_score(y_te, y_pred):.3f}")

Samples: 600, failure rate: 0.40, train/test: 420/180
Accuracy : 0.794
Precision: 0.783
Recall   : 0.662


In [3]:
# --- features of the 10 network nodes and predicted failure probabilities ---
node_X = np.column_stack([
    rng.uniform(1, 50, N), rng.uniform(0.3, 1.0, N),
    rng.uniform(0.0, 1.0, N), rng.uniform(5, 100, N)])
p_fail = clf.predict_proba(node_X)[:, 1]
print(pd.DataFrame({"node": range(N), "p_fail": p_fail.round(3)}).to_string(index=False))

# --- connected undirected network: ring (guarantees connectivity) + 8 random chords ---
edges = {(i, (i + 1) % N) if i < (i + 1) % N else ((i + 1) % N, i) for i in range(N)}
non_edges = [(i, j) for i in range(N) for j in range(i + 1, N) if (i, j) not in edges]
for k in rng.choice(len(non_edges), size=8, replace=False):
    edges.add(non_edges[k])
edge_list = sorted(edges)

# --- weights: w_ij = 1 + 10 * (p_i + p_j) / 2 ---
W = np.zeros((N, N))
for i, j in edge_list:
    W[i, j] = W[j, i] = 1 + 10 * (p_fail[i] + p_fail[j]) / 2

# connectivity check (BFS)
seen, stack = {0}, [0]
while stack:
    u = stack.pop()
    for v in np.nonzero(W[u])[0]:
        if int(v) not in seen:
            seen.add(int(v)); stack.append(int(v))
assert len(seen) == N, "Network is not connected"
assert W.shape == (N, N) and np.allclose(W, W.T) and (W >= 0).all()
print(f"\nNetwork: {N} nodes, {len(edge_list)} edges, connected = True")
print(pd.DataFrame([(i, j, round(W[i, j], 3)) for i, j in edge_list],
                   columns=["node_i", "node_j", "weight"]).to_string(index=False))

 node  p_fail
    0   0.439
    1   0.444
    2   0.293
    3   0.050
    4   0.650
    5   0.032
    6   0.228
    7   0.604
    8   0.122
    9   0.458

Network: 10 nodes, 18 edges, connected = True
 node_i  node_j  weight
      0       1   5.417
      0       6   4.339
      0       7   6.219
      0       9   5.486
      1       2   4.685
      1       5   3.382
      2       3   2.715
      2       9   4.754
      3       4   4.502
      3       8   1.861
      3       9   3.539
      4       5   4.414
      5       6   2.303
      5       9   3.451
      6       7   5.164
      6       8   2.752
      7       8   4.633
      8       9   3.900


# Part B: QUBO (1 Mark)
Binary variable $x_i \in \{0,1\}$: 0 = Island A, 1 = Island B.

$$E(x)=\sum_{i<j} w_{ij}(x_i-x_j)^2+\lambda\Big(\sum_i x_i-5\Big)^2$$

**Expansion** (using $x_i^2=x_i$):
- $(x_i-x_j)^2 = x_i + x_j - 2x_ix_j$, so the cut term gives linear coefficient $d_i=\sum_j w_{ij}$ and pair coefficient $-2w_{ij}$.
- $(\sum x_i-5)^2 = \sum_i(1-10)x_i + 2\sum_{i<j}x_ix_j + 25$.

**Convention:** $E(x)=x^\top Q x + \text{constant}$ with $Q$ symmetric, so  
$Q_{ii}=d_i-9\lambda$, $\quad Q_{ij}=Q_{ji}=-w_{ij}+\lambda$ (for $i\ne j$; $w_{ij}=0$ if no link), $\quad \text{constant}=25\lambda$.

**Choice of λ:** any configuration with $\sum x\ne5$ has penalty $\ge\lambda$ and cut $\ge0$. Any valid 5–5 split has energy $\le W_{total}$ (sum of all link weights). Choosing $\lambda = W_{total}+1$ guarantees every invalid configuration has higher energy than the optimal valid one.

In [4]:
W_total = sum(W[i, j] for i, j in edge_list)
lam = W_total + 1.0                      # penalty strength (see explanation above)

deg = W.sum(axis=1)                      # d_i = sum_j w_ij
Q = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        Q[i, j] = (deg[i] - (2*HALF - 1) * lam) if i == j else (-W[i, j] + lam)
const = HALF**2 * lam                    # constant offset = 25 * lambda

assert Q.shape == (N, N) and np.allclose(Q, Q.T)
print(f"W_total = {W_total:.3f}, lambda = {lam:.3f}, constant offset = {const:.3f}")
print("Convention: E(x) = x^T Q x + constant")
print("\nQ matrix (10 x 10):")
print(pd.DataFrame(Q.round(3)).to_string())

W_total = 73.518, lambda = 74.518, constant offset = 1862.949
Convention: E(x) = x^T Q x + constant

Q matrix (10 x 10):
         0        1        2        3        4        5        6        7        8        9
0 -649.200   69.101   74.518   74.518   74.518   74.518   70.179   68.299   74.518   69.032
1   69.101 -657.178   69.833   74.518   74.518   71.136   74.518   74.518   74.518   74.518
2   74.518   69.833 -658.507   71.803   74.518   74.518   74.518   74.518   74.518   69.763
3   74.518   74.518   71.803 -658.044   70.016   74.518   74.518   74.518   72.657   70.979
4   74.518   74.518   74.518   70.016 -661.746   70.104   74.518   74.518   74.518   74.518
5   74.518   71.136   74.518   74.518   70.104 -657.112   72.215   74.518   74.518   71.067
6   70.179   74.518   74.518   74.518   74.518   72.215 -656.103   69.354   71.766   74.518
7   68.299   74.518   74.518   74.518   74.518   74.518   69.354 -654.646   69.885   74.518
8   74.518   74.518   74.518   72.657   74.518   74

In [5]:
# --- energy functions ---
def energy_original(x):
    """Original objective: cut weight + lambda * (sum(x) - 5)^2."""
    x = np.asarray(x)
    cut = sum(W[i, j] * (x[i] - x[j])**2 for i, j in edge_list)
    return cut + lam * (x.sum() - HALF)**2

def energy_qubo(x):
    """QUBO form: x^T Q x + constant."""
    x = np.asarray(x, dtype=float)
    return x @ Q @ x + const

# --- all 2^10 = 1024 configurations (row k = binary representation of k) ---
ALL_X = ((np.arange(2**N)[:, None] >> np.arange(N)) & 1).astype(int)
E_orig = np.array([energy_original(x) for x in ALL_X])
E_qubo = np.array([energy_qubo(x) for x in ALL_X])

assert ALL_X.shape == (1024, N)
assert np.allclose(E_orig, E_qubo), "QUBO energy disagrees with original objective"
print("Check passed: QUBO energy == original objective for all 1024 configurations.")

Check passed: QUBO energy == original objective for all 1024 configurations.


# Part C: Quantum Computing (1.5 Marks)
Simulated quantum annealing via `dwave-neal` (a **classical CPU simulation**, not real quantum hardware) on the QUBO from Part B, compared with exhaustive brute force over all 1024 configurations. Swapping Island A and Island B gives the same partition (x and 1−x have equal energy), so partitions are compared up to this label swap.

In [6]:
# --- build dimod BQM (offset included, so energies equal the full objective) ---
linear = {i: float(Q[i, i]) for i in range(N)}
quadratic = {(i, j): float(2 * Q[i, j]) for i in range(N) for j in range(i + 1, N)}
bqm = dimod.BinaryQuadraticModel(linear, quadratic, float(const), dimod.BINARY)

# --- simulated annealing (fixed seed, many reads) ---
sampler = neal.SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm, num_reads=2000, num_sweeps=1000, seed=SEED).aggregate()

rows = []
for sample, energy, occ in sampleset.data(["sample", "energy", "num_occurrences"]):
    x = tuple(int(sample[i]) for i in range(N))
    rows.append((x, float(energy), int(occ)))
valid_rows = [r for r in rows if sum(r[0]) == HALF]
assert len(valid_rows) > 0, "Annealer returned no valid 5-5 solution"
sa_x, sa_E, sa_count = min(valid_rows, key=lambda r: r[1])
sa_comp = tuple(1 - b for b in sa_x)
sa_count_both = sum(r[2] for r in rows if r[0] in (sa_x, sa_comp))
total_reads = sum(r[2] for r in rows)
assert abs(sa_E - energy_original(sa_x)) < 1e-6 and sum(sa_x) == HALF

print(f"Reads: {total_reads}, valid-solution reads: {sum(r[2] for r in valid_rows)}")
print(f"Best valid annealing solution : {''.join(map(str, sa_x))}")
print(f"  Energy                      : {sa_E:.4f}")
print(f"  Appears (exact bitstring)   : {sa_count} / {total_reads} reads")
print(f"  Appears (incl. label swap)  : {sa_count_both} / {total_reads} reads")

Reads: 2000, valid-solution reads: 2000
Best valid annealing solution : 0111110000
  Energy                      : 21.3259
  Appears (exact bitstring)   : 16 / 2000 reads
  Appears (incl. label swap)  : 37 / 2000 reads


In [7]:
# --- brute force over all 1024 configurations (using the original objective) ---
valid_mask = ALL_X.sum(axis=1) == HALF
assert valid_mask.sum() == 252                       # C(10,5) valid configurations
E_valid = np.where(valid_mask, E_orig, np.inf)
bf_E = E_valid.min()
bf_idx = np.where(np.isclose(E_valid, bf_E))[0]      # all optimal configurations (incl. label swaps)
bf_x = tuple(int(b) for b in ALL_X[bf_idx[0]])
# canonical form: node 0 always in Island A
canon = lambda x: x if x[0] == 0 else tuple(1 - b for b in x)
bf_partitions = {canon(tuple(int(b) for b in ALL_X[k])) for k in bf_idx}
assert all(sum(p) == HALF for p in bf_partitions)

island_A = [i for i in range(N) if canon(bf_x)[i] == 0]
island_B = [i for i in range(N) if canon(bf_x)[i] == 1]
print(f"Brute-force optimum energy : {bf_E:.4f}")
print(f"Optimal 5-5 partition(s)   : {len(bf_partitions)} (up to label swap)")
print(f"  Island A = {island_A}\n  Island B = {island_B}")

# --- comparison ---
sa_matches = canon(sa_x) in bf_partitions and np.isclose(sa_E, bf_E)
print("\n--- Annealing vs brute force ---")
print(f"Annealing energy {sa_E:.4f} | Brute-force energy {bf_E:.4f}")
print("Annealing result MATCHES the brute-force optimum." if sa_matches
      else "Annealing result does NOT match the brute-force optimum (heuristic; try more reads/sweeps).")

Brute-force optimum energy : 21.3259
Optimal 5-5 partition(s)   : 1 (up to label swap)
  Island A = [0, 6, 7, 8, 9]
  Island B = [1, 2, 3, 4, 5]

--- Annealing vs brute force ---
Annealing energy 21.3259 | Brute-force energy 21.3259
Annealing result MATCHES the brute-force optimum.


# Part D: Parallel Computing (1 Mark)
All 1024 configurations are split into equal chunks (1024 divides evenly by 1, 2 and 4) and evaluated with `ProcessPoolExecutor` using 1, 2 and 4 workers.

**Colab note:** functions defined inside notebook cells cannot always be pickled for worker processes, so the worker function is written to an importable temporary module (`ot_worker.py`).

Speed-up = T1 / Tp, Efficiency = Speed-up / p. Timings include process start-up, so for such a small problem (only 1024 cheap evaluations) overhead may dominate and parallel runs may be **slower** than 1 worker. Results below are measured, not assumed.

In [8]:
from concurrent.futures import ProcessPoolExecutor

# --- worker module (importable by child processes) ---
worker_code = '''
def eval_chunk(args):
    start, end, n, edges, lam, half = args
    best_e, best_k = float("inf"), -1
    for k in range(start, end):
        x = [(k >> b) & 1 for b in range(n)]
        e = sum(w * (x[i] - x[j]) ** 2 for i, j, w in edges) + lam * (sum(x) - half) ** 2
        if e < best_e:
            best_e, best_k = e, k
    return best_e, best_k
'''
with open("ot_worker.py", "w") as f:
    f.write(worker_code)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
importlib.invalidate_caches()
import ot_worker
importlib.reload(ot_worker)

edges_w = [(i, j, float(W[i, j])) for i, j in edge_list]
TOTAL = 2**N

def run_parallel(p):
    """Evaluate all configurations with p workers; return (min energy, argmin index, time)."""
    size = TOTAL // p
    tasks = [(c*size, (c+1)*size, N, edges_w, float(lam), HALF) for c in range(p)]
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=p) as ex:
        results = list(ex.map(ot_worker.eval_chunk, tasks))
    t = time.perf_counter() - t0
    e, k = min(results)
    return e, k, t

workers = [1, 2, 4]
records, energies = [], {}
for p in workers:
    runs = [run_parallel(p) for _ in range(3)]          # best of 3 runs to reduce timing noise
    e, k, t = min(runs, key=lambda r: r[2])
    energies[p] = e
    records.append({"workers": p, "time_s": t})

T1 = records[0]["time_s"]
for r in records:
    r["speedup"] = T1 / r["time_s"]
    r["efficiency"] = r["speedup"] / r["workers"]
perf = pd.DataFrame(records)
print(perf.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# --- consistency checks: min over ALL 1024 configs (incl. penalty) must equal brute-force optimum ---
for p in workers:
    assert np.isclose(energies[p], bf_E), f"Mismatch for {p} workers"
assert np.isclose(E_orig.min(), bf_E)
print(f"\nAll worker counts give the same minimum energy: {bf_E:.4f}  (matches brute force)")

 workers  time_s  speedup  efficiency
       1  0.0290   1.0000      1.0000
       2  0.0538   0.5386      0.2693
       4  0.0920   0.3147      0.0787

All worker counts give the same minimum energy: 21.3259  (matches brute force)


# Part E: Conclusion (0.5 Marks)

In [9]:
sp = perf.set_index("workers")
print(f"1. Best partition: Island A = {island_A}, Island B = {island_B} with minimum energy (cut weight) = {bf_E:.4f}.")
print(f"2. Simulated annealing (CPU) found energy {sa_E:.4f} vs brute-force {bf_E:.4f}: "
      f"{'it matched the exact optimum in this run' if sa_matches else 'it did not match the exact optimum in this run'} (heuristic, no guarantee).")
print(f"3. All worker counts gave the same minimum; measured speed-up was {sp.loc[2,'speedup']:.2f}x (2 workers) and {sp.loc[4,'speedup']:.2f}x (4 workers), "
      f"so {'parallelism helped' if sp['speedup'].max() > 1.0 + 1e-9 else 'process overhead outweighed any gain'} for this small problem.")

1. Best partition: Island A = [0, 6, 7, 8, 9], Island B = [1, 2, 3, 4, 5] with minimum energy (cut weight) = 21.3259.
2. Simulated annealing (CPU) found energy 21.3259 vs brute-force 21.3259: it matched the exact optimum in this run (heuristic, no guarantee).
3. All worker counts gave the same minimum; measured speed-up was 0.54x (2 workers) and 0.31x (4 workers), so process overhead outweighed any gain for this small problem.
